# Platform Shift Analysis: Desktop vs. Mobile

This notebook analyzes the evolution of Wikipedia traffic across different access methods: Desktop, Mobile Web, and Mobile App.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath('../../src'))
from api_client import WikipediaAPIClient

In [ ]:
client = WikipediaAPIClient()
start_date = "20160101"
end_date = "20240101"
project = "en.wikipedia.org"

print("Fetching desktop pageviews...")
desktop = client.get_aggregate_pageviews(project, start_date, end_date, access="desktop")
print("Fetching mobile-web pageviews...")
mobile_web = client.get_aggregate_pageviews(project, start_date, end_date, access="mobile-web")
print("Fetching mobile-app pageviews...")
mobile_app = client.get_aggregate_pageviews(project, start_date, end_date, access="mobile-app")

In [ ]:
# Merge dataframes
desktop.rename(columns={'views': 'desktop'}, inplace=True)
mobile_web.rename(columns={'views': 'mobile_web'}, inplace=True)
mobile_app.rename(columns={'views': 'mobile_app'}, inplace=True)

df = desktop.merge(mobile_web, on='timestamp', how='outer').merge(mobile_app, on='timestamp', how='outer')
df['total_mobile'] = df['mobile_web'] + df['mobile_app']
df.head()

In [ ]:
# Visualization
plt.figure(figsize=(14, 7))
plt.plot(df['timestamp'], df['desktop'], label='Desktop', linewidth=2)
plt.plot(df['timestamp'], df['mobile_web'], label='Mobile Web', linewidth=2)
plt.plot(df['timestamp'], df['mobile_app'], label='Mobile App', linewidth=2)

plt.title(f'Wikipedia Pageviews by Access Method ({project})', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Views', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Share Analysis
df['total'] = df['desktop'] + df['total_mobile']
df['mobile_share'] = df['total_mobile'] / df['total']
df['desktop_share'] = df['desktop'] / df['total']

plt.figure(figsize=(14, 7))
plt.stackplot(df['timestamp'], df['desktop_share'], df['mobile_share'], labels=['Desktop Share', 'Mobile Share'], alpha=0.8)
plt.title('Share of Wikipedia Traffic: Desktop vs. Mobile', fontsize=16)
plt.ylabel('Proportion of Traffic')
plt.legend(loc='upper left')
plt.show()